In [25]:
import pandas as pd
import hashlib
from pathlib import Path

## 1 Leer archivos

# Clientes
clientes = pd.read_csv("/home/jovyan/datasets/Data_Lab03/clientes.csv");

# Ventas
ventas = pd.read_excel("/home/jovyan/datasets/Data_Lab03/ventas.xlsx", sheet_name="ventas")

#Contacto
contacto = pd.read_json("/home/jovyan/datasets/Data_Lab03/contacto.json")
print("Columnas de clientes:", clientes.columns.tolist())
print("Columnas de ventas:", ventas.columns.tolist())
print("Columnas de contacto:", contacto.columns.tolist())

## Paso 2 - Transformar la información
registros_antes = len(clientes);
clientes = clientes.drop_duplicates();
duplicados = registros_antes - len(clientes);

 #Normalizar nombre y ciudad ejemplos dato = seGOVIA (dato = Segovia).
clientes['nombre'] = clientes['nombre'].str.strip().str.capitalize();
clientes['ciudad'] = clientes['ciudad'].str.strip().str.title();

#Convertir fecha

clientes['fecha_alta'] = pd.to_datetime(
    clientes['fecha_alta'], format="%d/%m/%Y", errors="coerce"
).dt.strftime("%Y-%m-%d");


#Renombra la clave

ventas = ventas.rename(columns={"cliente_id": "id_cliente"})
contacto = contacto.rename(columns={"cliente_id": "id_cliente"})

# 3 Integrar la información
integrado = clientes.merge(ventas, on="id_cliente", how="left");
integrado = integrado.merge(contacto, on="id_cliente", how="left");

#Ordena y tipos de salida

integrado = integrado.sort_values(['id_cliente', "producto"], na_position="last")
for columna in ("id_venta", "cantidad"):
    integrado["cantidad"] = integrado["cantidad"].astype("Int64")


#4 Detectar incidencias
ciudades = clientes["ciudad"].fillna("").str.strip()

incidencias = {
    "duplicados": duplicados,
    "ciudades_vacias": (ciudades == "").sum(),
    "ventas_sin_cliente": (
        ~ventas["id_cliente"].isin(clientes["id_cliente"])
    ).sum(),
    "clientes_sin_correo": (
        ~clientes["id_cliente"].isin(contacto["id_cliente"])
    ).sum(),
    "contactos_sin_correspondencia": (
        ~contacto["id_cliente"].isin(clientes["id_cliente"])
    ).sum(),
}


#6 Guardar el resultado.
archivo = Path("resultadoLab03.csv");
integrado.to_csv(archivo,index=False,encoding="utf-8",lineterminator="\n");
digest = hashlib.sha256(archivo.read_bytes()).hexdigest()
print("Archivo guardado correctamente.")
print("SHA-256:", digest)
#7 Comprobar la reproducibilidad

Columnas de clientes: ['id_cliente', 'nombre', 'ciudad', 'fecha_alta']
Columnas de ventas: ['cliente_id', 'producto', 'cantidad', 'importe']
Columnas de contacto: ['cliente_id', 'email']
Archivo guardado correctamente.
SHA-256: 2b21d561ed384228ebf1cb1b065add4d0124409aa49fe3cf530978ef60187aaf
